# 1. Импорт библиотек и настройки

In [1]:
import pandas as pd

pd.set_option('display.float_format', '{:.4f}'.format)
from pprint import pprint  # noqa: F401

import numpy as np


# 2. Загрузка констант и динамики

In [2]:
const = {'Привлекаемые_средства': 380_000_000_000,
            'Ставка_купона_ОФЗ_ИН_л': 0.025,
            'Ставка_купона_ОФЗ_ПД': 0.1374,
            'Номинал_ОФЗ_ИН': 10_000,
            'Номинал_ОФЗ_ПД': 1000,
            'Количество_человек': 2_000_000,
            'НДФЛ': 0.13
        }
const

{'Привлекаемые_средства': 380000000000,
 'Ставка_купона_ОФЗ_ИН_л': 0.025,
 'Ставка_купона_ОФЗ_ПД': 0.1374,
 'Номинал_ОФЗ_ИН': 10000,
 'Номинал_ОФЗ_ПД': 1000,
 'Количество_человек': 2000000,
 'НДФЛ': 0.13}

In [3]:
# загружаем функии из модуля cbr_inflation
from datetime import date

import function.cbr_inflation as cbr_inf

# загружаем из модуля new_function
from function.api_in_function import get_deposit_rates

inf = cbr_inf.get_inflation()

# выбираем только последний год и последнее значение инфляции
inf_d = inf.copy()
inf_d['Год'] = inf_d['date'].dt.year
inf_d.rename(columns={
    'inflation': 'Инфляция',
    'target': 'Цель по инфляции'},inplace=True)
inf_d = inf_d[['Год','Инфляция']]
inf_d = inf_d.tail(1)
# автоматизируем продлжения ряда лет,и настраиваем вывод целовой инфляции
current_year = date.today().year  # noqa: DTZ011
forecast_years = [ current_year + 1, current_year + 2]
inf2 = pd.DataFrame({
    'Год': forecast_years,
    'Инфляция': inf['target'].iloc[-1]})
inf_res = pd.concat([inf_d,inf2],ignore_index=True)

In [4]:
[inf['target'].iloc[-1]]

[np.float64(4.0)]

In [5]:
df = get_deposit_rates()
sd = df.tail(1)
# создаем переменную имеющую единственную последнюю актуальную ставку
value = sd['rate'].iloc[0]

✅ Запрос успешен. Первые 200 символов: '{"RawData":[{"indicator_id":37,"measure_1_id":2,"measure_2_id":7,"unit_id":1,"value":4.4600,"period_id":337,"period":"Январь 2020","periodicity":"month","date":"01.02.2020","rowId":8114487},{"indicato'


In [6]:
DEPOSIT_DECREMENT = 2.5 # коэфициент снижения 
base = value
inf_res['Ставка депозита'] = base - DEPOSIT_DECREMENT * inf_res.index
inf_res[['Инфляция','Ставка депозита']] = inf_res[['Инфляция','Ставка депозита']]/100 # переводим проценты в числа
inf_res

,Год,Инфляция,Ставка депозита
0,2026,0.0633,0.1281
1,2027,0.0400,0.1031
2,2028,0.0400,0.0781


# 3. ОФЗ ИН (л)

In [7]:
ofz_in_l =inf_res.copy()
ofz_in_l

,Год,Инфляция,Ставка депозита
0,2026,0.0633,0.1281
1,2027,0.0400,0.1031
2,2028,0.0400,0.0781


In [8]:
ofz_in_l['Привлекаемые средства'] = const['Привлекаемые_средства']
ofz_in_l['Количество человек'] = const['Количество_человек']
ofz_in_l['Ставка купона'] = const['Ставка_купона_ОФЗ_ИН_л']
ofz_in_l

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона
0,2026,0.0633,0.1281,380000000000,2000000,0.0250
1,2027,0.0400,0.1031,380000000000,2000000,0.0250
2,2028,0.0400,0.0781,380000000000,2000000,0.0250


In [9]:
ofz_in_l['На руках у человека, руб'] = ofz_in_l['Привлекаемые средства'] / ofz_in_l['Количество человек']
ofz_in_l

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб"
0,2026,0.0633,0.1281,380000000000,2000000,0.0250,190000.0000
1,2027,0.0400,0.1031,380000000000,2000000,0.0250,190000.0000
2,2028,0.0400,0.0781,380000000000,2000000,0.0250,190000.0000


In [10]:
ofz_in_l['Облигаций штук'] = ofz_in_l['На руках у человека, руб'] / const ['Номинал_ОФЗ_ИН']
ofz_in_l

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук
0,2026,0.0633,0.1281,380000000000,2000000,0.0250,190000.0000,19.0000
1,2027,0.0400,0.1031,380000000000,2000000,0.0250,190000.0000,19.0000
2,2028,0.0400,0.0781,380000000000,2000000,0.0250,190000.0000,19.0000


In [11]:
ofz_in_l['Инфляционный множитель']= (1 + inf_res['Инфляция']).cumprod()
ofz_in_l

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель
0,2026,0.0633,0.1281,380000000000,2000000,0.0250,190000.0000,19.0000,1.0633
1,2027,0.0400,0.1031,380000000000,2000000,0.0250,190000.0000,19.0000,1.1058
2,2028,0.0400,0.0781,380000000000,2000000,0.0250,190000.0000,19.0000,1.1501


In [12]:
ofz_in_l['Номинал после индексации'] = const['Номинал_ОФЗ_ИН']*ofz_in_l['Инфляционный множитель']
ofz_in_l

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации
0,2026,0.0633,0.1281,380000000000,2000000,0.0250,190000.0000,19.0000,1.0633,10633.0000
1,2027,0.0400,0.1031,380000000000,2000000,0.0250,190000.0000,19.0000,1.1058,11058.3200
2,2028,0.0400,0.0781,380000000000,2000000,0.0250,190000.0000,19.0000,1.1501,11500.6528


In [13]:
ofz_in_l['Номинал на начало'] = float(const['Номинал_ОФЗ_ИН'])
ofz_in_l.loc[ofz_in_l.index > 0, 'Номинал на начало'] = ofz_in_l['Номинал после индексации'].shift(1).fillna(const['Номинал_ОФЗ_ИН'])
ofz_in_l

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации,Номинал на начало
0,2026,0.0633,0.1281,380000000000,2000000,0.0250,190000.0000,19.0000,1.0633,10633.0000,10000.0000
1,2027,0.0400,0.1031,380000000000,2000000,0.0250,190000.0000,19.0000,1.1058,11058.3200,10633.0000
2,2028,0.0400,0.0781,380000000000,2000000,0.0250,190000.0000,19.0000,1.1501,11500.6528,11058.3200


In [14]:
ofz_in_l['Индексация номинала'] = ofz_in_l['Номинал на начало'] * ofz_in_l['Инфляция']
ofz_in_l

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации,Номинал на начало,Индексация номинала
0,2026,0.0633,0.1281,380000000000,2000000,0.0250,190000.0000,19.0000,1.0633,10633.0000,10000.0000,633.0000
1,2027,0.0400,0.1031,380000000000,2000000,0.0250,190000.0000,19.0000,1.1058,11058.3200,10633.0000,425.3200
2,2028,0.0400,0.0781,380000000000,2000000,0.0250,190000.0000,19.0000,1.1501,11500.6528,11058.3200,442.3328


In [15]:
ofz_in_l['Купон, руб'] = ofz_in_l['Номинал после индексации'] * ofz_in_l['Ставка купона']
ofz_in_l

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации,Номинал на начало,Индексация номинала,"Купон, руб"
0,2026,0.0633,0.1281,380000000000,2000000,0.0250,190000.0000,19.0000,1.0633,10633.0000,10000.0000,633.0000,265.8250
1,2027,0.0400,0.1031,380000000000,2000000,0.0250,190000.0000,19.0000,1.1058,11058.3200,10633.0000,425.3200,276.4580
2,2028,0.0400,0.0781,380000000000,2000000,0.0250,190000.0000,19.0000,1.1501,11500.6528,11058.3200,442.3328,287.5163


In [16]:
ofz_in_l['Доход без вычета'] = ofz_in_l['Купон, руб'] * ofz_in_l['Облигаций штук']
ofz_in_l

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации,Номинал на начало,Индексация номинала,"Купон, руб",Доход без вычета
0,2026,0.0633,0.1281,380000000000,2000000,0.0250,190000.0000,19.0000,1.0633,10633.0000,10000.0000,633.0000,265.8250,5050.6750
1,2027,0.0400,0.1031,380000000000,2000000,0.0250,190000.0000,19.0000,1.1058,11058.3200,10633.0000,425.3200,276.4580,5252.7020
2,2028,0.0400,0.0781,380000000000,2000000,0.0250,190000.0000,19.0000,1.1501,11500.6528,11058.3200,442.3328,287.5163,5462.8101


In [17]:
ofz_in_l ['Налоговый вычет, руб']= ofz_in_l['На руках у человека, руб'] * const['НДФЛ']
ofz_in_l

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации,Номинал на начало,Индексация номинала,"Купон, руб",Доход без вычета,"Налоговый вычет, руб"
0,2026,0.0633,0.1281,380000000000,2000000,0.0250,190000.0000,19.0000,1.0633,10633.0000,10000.0000,633.0000,265.8250,5050.6750,24700.0000
1,2027,0.0400,0.1031,380000000000,2000000,0.0250,190000.0000,19.0000,1.1058,11058.3200,10633.0000,425.3200,276.4580,5252.7020,24700.0000
2,2028,0.0400,0.0781,380000000000,2000000,0.0250,190000.0000,19.0000,1.1501,11500.6528,11058.3200,442.3328,287.5163,5462.8101,24700.0000


In [18]:
ofz_in_l['Доход с вычетом'] = ofz_in_l['Налоговый вычет, руб'] + ofz_in_l['Доход без вычета']
ofz_in_l

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации,Номинал на начало,Индексация номинала,"Купон, руб",Доход без вычета,"Налоговый вычет, руб",Доход с вычетом
0,2026,0.0633,0.1281,380000000000,2000000,0.0250,190000.0000,19.0000,1.0633,10633.0000,10000.0000,633.0000,265.8250,5050.6750,24700.0000,29750.6750
1,2027,0.0400,0.1031,380000000000,2000000,0.0250,190000.0000,19.0000,1.1058,11058.3200,10633.0000,425.3200,276.4580,5252.7020,24700.0000,29952.7020
2,2028,0.0400,0.0781,380000000000,2000000,0.0250,190000.0000,19.0000,1.1501,11500.6528,11058.3200,442.3328,287.5163,5462.8101,24700.0000,30162.8101


In [19]:
ofz_in_l = ofz_in_l[['Год', # перезаписываем в нужном порядке
 'Привлекаемые средства',
 'Количество человек',
 'Инфляция',
 'Ставка купона',
 'На руках у человека, руб',
 'Облигаций штук',
 'Инфляционный множитель',
 'Номинал на начало',
 'Индексация номинала',
 'Номинал после индексации',
 'Купон, руб',
 'Доход без вычета',
 'Налоговый вычет, руб',
 'Доход с вычетом']]
ofz_in_l

,Год,Привлекаемые средства,Количество человек,Инфляция,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал на начало,Индексация номинала,Номинал после индексации,"Купон, руб",Доход без вычета,"Налоговый вычет, руб",Доход с вычетом
0,2026,380000000000,2000000,0.0633,0.0250,190000.0000,19.0000,1.0633,10000.0000,633.0000,10633.0000,265.8250,5050.6750,24700.0000,29750.6750
1,2027,380000000000,2000000,0.0400,0.0250,190000.0000,19.0000,1.1058,10633.0000,425.3200,11058.3200,276.4580,5252.7020,24700.0000,29952.7020
2,2028,380000000000,2000000,0.0400,0.0250,190000.0000,19.0000,1.1501,11058.3200,442.3328,11500.6528,287.5163,5462.8101,24700.0000,30162.8101


# 4. ОФЗ ПД

In [20]:
ofz_pd = ofz_in_l [['Год']].copy()
ofz_pd['Привлекаемые средства'] = const ["Привлекаемые_средства"]
ofz_pd ["Количество человек"] = const ["Количество_человек"]
ofz_pd ["Ставка купона"] = const ['Ставка_купона_ОФЗ_ПД']
ofz_pd ["На руках у человека"] = ofz_in_l [["На руках у человека, руб"]].copy()
ofz_pd ['Облигаций, штук'] = ofz_pd ['На руках у человека'] / const ['Номинал_ОФЗ_ПД']
ofz_pd ['Купон'] = const ['Номинал_ОФЗ_ПД'] * const ['Ставка_купона_ОФЗ_ПД']
ofz_pd ["Доход, руб"] = ofz_pd ["Купон"] * ofz_pd ['Облигаций, штук']
ofz_pd ['НДФЛ'] = ofz_pd ['Доход, руб'] * const ['НДФЛ']
ofz_pd ['Доход после вычета налога'] = ofz_pd['Доход, руб'] - ofz_pd ['НДФЛ']
ofz_pd

,Год,Привлекаемые средства,Количество человек,Ставка купона,На руках у человека,"Облигаций, штук",Купон,"Доход, руб",НДФЛ,Доход после вычета налога
0,2026,380000000000,2000000,0.1374,190000.0000,190.0000,137.4000,26106.0000,3393.7800,22712.2200
1,2027,380000000000,2000000,0.1374,190000.0000,190.0000,137.4000,26106.0000,3393.7800,22712.2200
2,2028,380000000000,2000000,0.1374,190000.0000,190.0000,137.4000,26106.0000,3393.7800,22712.2200


# 5. Депозит

In [21]:
depozit = ofz_in_l[['Год']].copy()
depozit ['Привлекаемые средства'] = const ['Привлекаемые_средства']
depozit ['Количество человек'] = const ['Количество_человек']
depozit ['На руках у человека'] = ofz_in_l ['На руках у человека, руб']
depozit ['Ставка депозита'] = inf_res['Ставка депозита'] 


In [22]:
depozit ['Коэфициент'] = (1 + inf_res['Ставка депозита'] / 12) **12
depozit

,Год,Привлекаемые средства,Количество человек,На руках у человека,Ставка депозита,Коэфициент
0,2026,380000000000,2000000,190000.0000,0.1281,1.1359
1,2027,380000000000,2000000,190000.0000,0.1031,1.1081
2,2028,380000000000,2000000,190000.0000,0.0781,1.0810


In [23]:
depozit ['Накопленный множитель'] = depozit ['Коэфициент'].cumprod()
depozit

,Год,Привлекаемые средства,Количество человек,На руках у человека,Ставка депозита,Коэфициент,Накопленный множитель
0,2026,380000000000,2000000,190000.0000,0.1281,1.1359,1.1359
1,2027,380000000000,2000000,190000.0000,0.1031,1.1081,1.2587
2,2028,380000000000,2000000,190000.0000,0.0781,1.0810,1.3606


In [24]:
initial_amount = depozit['На руках у человека'].iloc[0]

In [25]:
depozit ['Сумма на конец года'] = initial_amount * depozit ['Накопленный множитель']
depozit

,Год,Привлекаемые средства,Количество человек,На руках у человека,Ставка депозита,Коэфициент,Накопленный множитель,Сумма на конец года
0,2026,380000000000,2000000,190000.0000,0.1281,1.1359,1.1359,215820.0947
1,2027,380000000000,2000000,190000.0000,0.1031,1.1081,1.2587,239153.3041
2,2028,380000000000,2000000,190000.0000,0.0781,1.0810,1.3606,258514.4865


In [26]:
depozit ['Сумма на начало года'] = initial_amount
depozit.loc [depozit.index > 0, 'Сумма на начало года'] = depozit['Сумма на конец года']. shift (1)
# Более хорошая альтернатива:
# depozit['Сумма на начало года '] = depozit['Сумма на конец года'].shift(1).fillna(depozit['На руках у человека, руб'].iloc[0])
#depozit['Сумма начало'] = depozit['Сумма конец'].shift(1).fillna(initial_amount)
depozit

,Год,Привлекаемые средства,Количество человек,На руках у человека,Ставка депозита,Коэфициент,Накопленный множитель,Сумма на конец года,Сумма на начало года
0,2026,380000000000,2000000,190000.0000,0.1281,1.1359,1.1359,215820.0947,190000.0000
1,2027,380000000000,2000000,190000.0000,0.1031,1.1081,1.2587,239153.3041,215820.0947
2,2028,380000000000,2000000,190000.0000,0.0781,1.0810,1.3606,258514.4865,239153.3041


In [27]:
depozit['Проценты']= depozit['Сумма на конец года'] - depozit['Сумма на начало года']
depozit

,Год,Привлекаемые средства,Количество человек,На руках у человека,Ставка депозита,Коэфициент,Накопленный множитель,Сумма на конец года,Сумма на начало года,Проценты
0,2026,380000000000,2000000,190000.0000,0.1281,1.1359,1.1359,215820.0947,190000.0000,25820.0947
1,2027,380000000000,2000000,190000.0000,0.1031,1.1081,1.2587,239153.3041,215820.0947,23333.2094
2,2028,380000000000,2000000,190000.0000,0.0781,1.0810,1.3606,258514.4865,239153.3041,19361.1824


In [28]:
depozit = depozit [["Год", "Привлекаемые средства", # перезаписываем в нужном порядке
                   "Количество человек", 
                   "На руках у человека",
                   "Ставка депозита",
                   "Коэфициент",
                   "Накопленный множитель",
                   "Сумма на начало года",
                   "Сумма на конец года",
                    "Проценты"]]
depozit          

,Год,Привлекаемые средства,Количество человек,На руках у человека,Ставка депозита,Коэфициент,Накопленный множитель,Сумма на начало года,Сумма на конец года,Проценты
0,2026,380000000000,2000000,190000.0000,0.1281,1.1359,1.1359,190000.0000,215820.0947,25820.0947
1,2027,380000000000,2000000,190000.0000,0.1031,1.1081,1.2587,215820.0947,239153.3041,23333.2094
2,2028,380000000000,2000000,190000.0000,0.0781,1.0810,1.3606,239153.3041,258514.4865,19361.1824


# 6. Доход за период (собрал без merge)

In [29]:
ofz_in_l_summary = pd.DataFrame({
    'Инструмент': ['ОФЗ ИН (л)'],
    'На руках у человека': [ofz_in_l['На руках у человека, руб'].iloc[0]],
    'Доход, до налогов': [ofz_in_l ['Индексация номинала'].sum()*ofz_in_l['Облигаций штук'].iloc[0] +  ofz_in_l['Доход без вычета'].sum()],
})
ofz_in_l_summary ['Итоговая сумма'] = ofz_in_l['На руках у человека, руб'].iloc[0] + ofz_in_l_summary['Доход, до налогов'] + ofz_in_l ['Налоговый вычет, руб'].iloc [0] 
ofz_in_l_summary

,Инструмент,На руках у человека,"Доход, до налогов",Итоговая сумма
0,ОФЗ ИН (л),190000.0000,44278.5903,258978.5903


In [30]:
ofz_pd_summary = pd.DataFrame({
    'Инструмент': ['ОФЗ ПД'],
    'На руках у человека': [ofz_in_l ['На руках у человека, руб'].iloc[0]],
    'Доход, до налогов': [ofz_pd['Доход, руб'].sum()],
    'Итоговая сумма': [ofz_in_l ['На руках у человека, руб'].iloc[0] + ofz_pd['Доход, руб'].sum()]})
ofz_pd_summary

,Инструмент,На руках у человека,"Доход, до налогов",Итоговая сумма
0,ОФЗ ПД,190000.0000,78318.0000,268318.0000


In [31]:
depozit_summary = pd.DataFrame({
    'Инструмент': ['Депозит'],
    'На руках у человека': [ofz_in_l ['На руках у человека, руб'].iloc[0]],
    'Доход, до налогов': [depozit['Проценты'].sum()],
    'Итоговая сумма': [ofz_in_l ['На руках у человека, руб'].iloc[0] + depozit['Проценты'].sum()]})
depozit_summary

,Инструмент,На руках у человека,"Доход, до налогов",Итоговая сумма
0,Депозит,190000.0000,68514.4865,258514.4865


In [32]:
svodnay_dont_merge = pd.concat([ofz_in_l_summary, ofz_pd_summary,depozit_summary],ignore_index= True)
svodnay_dont_merge

,Инструмент,На руках у человека,"Доход, до налогов",Итоговая сумма
0,ОФЗ ИН (л),190000.0000,44278.5903,258978.5903
1,ОФЗ ПД,190000.0000,78318.0000,268318.0000
2,Депозит,190000.0000,68514.4865,258514.4865


In [33]:
inf_factor = (1 + inf_res['Инфляция']).prod()
inf_factor

np.float64(1.15006528)

In [34]:
svodnay_dont_merge['Очистка инфляции'] = svodnay_dont_merge['Итоговая сумма']/inf_factor # pyright: ignore[reportOperatorIssue]
svodnay_dont_merge['Реальный доход'] = np.where(
    svodnay_dont_merge['Инструмент'] == 'ОФЗ ИН (л)',
    svodnay_dont_merge['Итоговая сумма'] - svodnay_dont_merge['На руках у человека'],
    svodnay_dont_merge['Очистка инфляции'] - svodnay_dont_merge['На руках у человека'])
svodnay_dont_merge.loc[svodnay_dont_merge['Инструмент'] == 'ОФЗ ИН (л)', 'Очистка инфляции'] = np.nan
svodnay_dont_merge

,Инструмент,На руках у человека,"Доход, до налогов",Итоговая сумма,Очистка инфляции,Реальный доход
0,ОФЗ ИН (л),190000.0000,44278.5903,258978.5903,NaN,68978.5903
1,ОФЗ ПД,190000.0000,78318.0000,268318.0000,233306.7563,43306.7563
2,Депозит,190000.0000,68514.4865,258514.4865,224782.4458,34782.4458


In [35]:
# svodnay_dont_merge['Очистка инфляции'] = svodnay_dont_merge['Итоговая сумма']/inf_factor # pyright: ignore[reportOperatorIssue]
# svodnay_dont_merge['Реальный доход'] = svodnay_dont_merge['Очистка инфляции'] - svodnay_dont_merge['На руках у человека']
# svodnay_dont_merge

In [36]:
# svodnay_dont_merge.loc[svodnay_dont_merge['Инструмент'] == 'ОФЗ ИН (л)', 'Очистка инфляции'] = None
# # pohti_konec.loc[pohti_konec['Инструмент'] == 'ОФЗ ИН (л)', 'Реальный доход'] = None
# svodnay_dont_merge

# Делаем таблицу с merge

In [37]:
df_s_merge = ofz_in_l[['Год', 'На руках у человека, руб']].copy()
df_s_merge.rename(columns={'На руках у человека, руб': 'Вложения'}, inplace=True)
df_s_merge

,Год,Вложения
0,2026,190000.0000
1,2027,190000.0000
2,2028,190000.0000


In [38]:
df_s_merge['ОФЗ ИН доход'] = ofz_in_l['Индексация номинала']* ofz_in_l['Облигаций штук'] + ofz_in_l['Доход без вычета']
df_s_merge

,Год,Вложения,ОФЗ ИН доход
0,2026,190000.0000,17077.6750
1,2027,190000.0000,13333.7820
2,2028,190000.0000,13867.1333


In [39]:
df_s_merge = df_s_merge.merge(ofz_pd[['Год', 'Доход, руб']], on='Год', how='left')
df_s_merge.rename(columns={'Доход, руб': 'ОФЗ ПД доход'}, inplace=True)
df_s_merge

,Год,Вложения,ОФЗ ИН доход,ОФЗ ПД доход
0,2026,190000.0000,17077.6750,26106.0000
1,2027,190000.0000,13333.7820,26106.0000
2,2028,190000.0000,13867.1333,26106.0000


In [40]:
df_s_merge = df_s_merge.merge(depozit[['Год', 'Проценты']], on='Год', how='left')
df_s_merge.rename(columns={'Проценты': 'Депозит доход'}, inplace=True)
df_s_merge

,Год,Вложения,ОФЗ ИН доход,ОФЗ ПД доход,Депозит доход
0,2026,190000.0000,17077.6750,26106.0000,25820.0947
1,2027,190000.0000,13333.7820,26106.0000,23333.2094
2,2028,190000.0000,13867.1333,26106.0000,19361.1824


# Делаем длинную 


In [41]:
# Теперь применяем melt к данным 
df_long = df_s_merge.melt(
    id_vars=['Год', 'Вложения'],
    value_vars=['ОФЗ ИН доход', 'ОФЗ ПД доход', 'Депозит доход'],
    var_name='Инструмент',
    value_name='Доход'
)

# Сортируем по году и инструменту
df_long = df_long.sort_values(['Год', 'Инструмент']).reset_index(drop=True)

df_long

,Год,Вложения,Инструмент,Доход
0,2026,190000.0000,Депозит доход,25820.0947
1,2026,190000.0000,ОФЗ ИН доход,17077.6750
2,2026,190000.0000,ОФЗ ПД доход,26106.0000
3,2027,190000.0000,Депозит доход,23333.2094
4,2027,190000.0000,ОФЗ ИН доход,13333.7820
5,2027,190000.0000,ОФЗ ПД доход,26106.0000
6,2028,190000.0000,Депозит доход,19361.1824
7,2028,190000.0000,ОФЗ ИН доход,13867.1333
8,2028,190000.0000,ОФЗ ПД доход,26106.0000


In [42]:
ofz_in_part = df_long[df_long['Инструмент'] == 'ОФЗ ИН доход'].groupby('Инструмент', as_index=False)['Доход'].sum()
ofz_in_part['Итоговая сумма'] = ofz_in_l['На руках у человека, руб'].iloc[0] + ofz_in_part.loc[ofz_in_part['Инструмент'] == 'ОФЗ ИН доход','Доход'] + ofz_in_l['Налоговый вычет, руб'].iloc[0]
ofz_in_part['Очистка инфляции'] = np.nan
ofz_in_part['Реальный доход'] = ofz_in_part['Итоговая сумма'] - ofz_in_l['На руках у человека, руб']
ofz_in_part

,Инструмент,Доход,Итоговая сумма,Очистка инфляции,Реальный доход
0,ОФЗ ИН доход,44278.5903,258978.5903,NaN,68978.5903


In [43]:
# doxod_za_period = pd.DataFrame()
drugoe = df_long[df_long['Инструмент'] !='ОФЗ ИН доход'].groupby('Инструмент', as_index=False)['Доход'].sum()
drugoe
# doxod_za_period =  df_long.groupby('Инструмент', as_index=False)['Доход'].sum()
# #agg(Итоговая_сумма= ('Доход', sum) 

# doxod_za_period

,Инструмент,Доход
0,Депозит доход,68514.4865
1,ОФЗ ПД доход,78318.0000


In [44]:
# Вычисляем итоговые суммы
# total_oin = ofz_in_l['На руках у человека, руб'].iloc[0] + doxod_za_period.loc[doxod_za_period['Инструмент'] == 'ОФЗ ИН доход','Доход'].iloc[0] + ofz_in_l['Налоговый вычет, руб'].iloc[0]
total_pd = ofz_in_l['На руках у человека, руб'].iloc[0] + ofz_pd['Доход, руб'].sum()
total_dep = ofz_in_l['На руках у человека, руб'].iloc[0] + depozit['Проценты'].sum()

# Словарь соответствий
total_dict = {
    # 'ОФЗ ИН доход': total_oin,
    'ОФЗ ПД доход': total_pd,
    'Депозит доход': total_dep
}

# Добавляем колонку Итоговая сумма
drugoe['Итоговая сумма'] = drugoe['Инструмент'].map(total_dict)
drugoe

,Инструмент,Доход,Итоговая сумма
0,Депозит доход,68514.4865,258514.4865
1,ОФЗ ПД доход,78318.0000,268318.0000


In [45]:
drugoe['Очистка инфляции'] = drugoe['Итоговая сумма']/inf_factor
drugoe['Реальный доход'] = drugoe['Итоговая сумма'] - ofz_in_l['На руках у человека, руб']
ofz_in_part['Очистка инфляции'] = np.nan
ofz_in_part['Реальный доход'] = ofz_in_part['Итоговая сумма'] - ofz_in_l['На руках у человека, руб']
ofz_in_part

,Инструмент,Доход,Итоговая сумма,Очистка инфляции,Реальный доход
0,ОФЗ ИН доход,44278.5903,258978.5903,NaN,68978.5903


In [46]:
drugoe

,Инструмент,Доход,Итоговая сумма,Очистка инфляции,Реальный доход
0,Депозит доход,68514.4865,258514.4865,224782.4458,68514.4865
1,ОФЗ ПД доход,78318.0000,268318.0000,233306.7563,78318.0000


In [47]:
test_itog = pd.concat([ofz_in_part,drugoe],ignore_index=True)
test_itog

,Инструмент,Доход,Итоговая сумма,Очистка инфляции,Реальный доход
0,ОФЗ ИН доход,44278.5903,258978.5903,NaN,68978.5903
1,Депозит доход,68514.4865,258514.4865,224782.4458,68514.4865
2,ОФЗ ПД доход,78318.0000,268318.0000,233306.7563,78318.0000


# Нагрузка на государство


In [48]:
ofz_in_l_gos = inf_res.copy()
ofz_in_l_gos['Прибавка от инфляции'] = const['Привлекаемые_средства']*ofz_in_l_gos['Инфляция']
ofz_in_l_gos['Инфляционный множитель'] = (1+ofz_in_l_gos ['Инфляция']).cumprod()
ofz_in_l_gos['Тело долга'] = const['Привлекаемые_средства'] * ofz_in_l_gos['Инфляционный множитель']
ofz_in_l_gos['Расходы на купоны'] = ofz_in_l_gos['Тело долга'] * const['Ставка_купона_ОФЗ_ИН_л']
ofz_in_l_gos['Общие затраты'] = ofz_in_l_gos['Прибавка от инфляции']+ofz_in_l_gos['Расходы на купоны']
ofz_in_l_gos = ofz_in_l_gos[['Год','Инфляция','Прибавка от инфляции','Тело долга','Расходы на купоны','Общие затраты']]
ofz_in_l_gos

,Год,Инфляция,Прибавка от инфляции,Тело долга,Расходы на купоны,Общие затраты
0,2026,0.0633,24054000000.0000,404053999999.9999,10101350000.0000,34155350000.0000
1,2027,0.0400,15200000000.0000,420216160000.0000,10505404000.0000,25705404000.0000
2,2028,0.0400,15200000000.0000,437024806400.0000,10925620160.0000,26125620160.0000


In [49]:
cols = ['Прибавка от инфляции', 'Тело долга', 'Расходы на купоны','Общие затраты']
total_ofz_in_l_gos = pd.DataFrame(ofz_in_l_gos[cols].sum()).T
total_ofz_in_l_gos['Год'] = 'Итого'
itog_ofz_in_l_gos = pd.concat([ofz_in_l_gos,total_ofz_in_l_gos],ignore_index=True)
itog_ofz_in_l_gos

,Год,Инфляция,Прибавка от инфляции,Тело долга,Расходы на купоны,Общие затраты
0,2026,0.0633,24054000000.0000,404053999999.9999,10101350000.0000,34155350000.0000
1,2027,0.0400,15200000000.0000,420216160000.0000,10505404000.0000,25705404000.0000
2,2028,0.0400,15200000000.0000,437024806400.0000,10925620160.0000,26125620160.0000
3,Итого,NaN,54454000000.0000,1261294966400.0000,31532374160.0000,85986374160.0000


In [50]:
ofz_pd_gos = ofz_in_l_gos[['Год']].copy()
ofz_pd_gos['Тело долга'] = const['Привлекаемые_средства']
ofz_pd_gos['Расходы на купон'] = ofz_pd_gos['Тело долга'] * const['Ставка_купона_ОФЗ_ПД']
ofz_pd_gos['Сумма возврата,НДФЛ'] = ofz_pd['НДФЛ']*const['Количество_человек']
ofz_pd_gos['Итого при учете возврата НДФЛ'] = ofz_pd_gos['Расходы на купон']-ofz_pd_gos['Сумма возврата,НДФЛ']
ofz_pd_gos

,Год,Тело долга,Расходы на купон,"Сумма возврата,НДФЛ",Итого при учете возврата НДФЛ
0,2026,380000000000,52212000000.0000,6787560000.0000,45424440000.0000
1,2027,380000000000,52212000000.0000,6787560000.0000,45424440000.0000
2,2028,380000000000,52212000000.0000,6787560000.0000,45424440000.0000


In [51]:
cols2 = ['Расходы на купон','Сумма возврата,НДФЛ','Итого при учете возврата НДФЛ']
total_ofz_pd_gos = pd.DataFrame([ofz_pd_gos[cols2].sum()])
total_ofz_pd_gos['Год'] = 'Итого'
itog_ofz_pd_gos = pd.concat([ofz_pd_gos,total_ofz_pd_gos],ignore_index=True)
itog_ofz_pd_gos

,Год,Тело долга,Расходы на купон,"Сумма возврата,НДФЛ",Итого при учете возврата НДФЛ
0,2026,380000000000.0000,52212000000.0000,6787560000.0000,45424440000.0000
1,2027,380000000000.0000,52212000000.0000,6787560000.0000,45424440000.0000
2,2028,380000000000.0000,52212000000.0000,6787560000.0000,45424440000.0000
3,Итого,NaN,156636000000.0000,20362680000.0000,136273320000.0000
